# Beating heart from gated cardiac CT

Every step behind the `/imaging/heart/` page, in the order it runs. Executed in WSL Ubuntu 26.04
on an RTX 4050 Laptop (6 GB), Python 3.12, torch cu126, monai-physio 2026.9.1.

The pipeline follows monai-physio Tutorial 1: a 4D cardiac CT is segmented on one reference
phase, every other phase is registered to it, and the resulting time-varying surfaces are
assembled into an animated model. The motion is measured from the scan, not simulated.

Environment: `~/heart/.venv`. Data: `~/heart/data`. Output: `~/heart/out`.

## 1. Environment

What is installed, and whether the GPU is reachable from WSL.

In [1]:
import importlib.metadata as meta
import os
import platform
from pathlib import Path

import torch

HOME = Path.home() / "heart"
DATA = HOME / "data"
OUT = HOME / "out"
# Where the dashboard looks for preview images, on the Windows side of the mount.
DASH = Path("/mnt/c/Users/vanio/claudeOS/projects/vanioAntunes/.tmp/heart")
for d in (DATA, OUT, DASH):
    d.mkdir(parents=True, exist_ok=True)

os.environ["MONAI_PHYSIO_INPUT_DATA_DIR"] = str(DATA)
os.environ["MONAI_PHYSIO_OUTPUT_DATA_DIR"] = str(OUT)

print("python     ", platform.python_version())
print("torch      ", torch.__version__, "| cuda", torch.version.cuda)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("gpu        ", props.name, f"{props.total_memory / 1024**3:.1f} GB")
else:
    print("gpu         none visible")

for pkg in ["monai-physio", "monai", "totalsegmentator", "vtk", "trimesh", "itk", "antspyx"]:
    try:
        print(f"{pkg:<17}", meta.version(pkg))
    except meta.PackageNotFoundError:
        print(f"{pkg:<17}", "not installed")

python      3.12.15
torch       2.14.1+cu126 | cuda 12.6
gpu         NVIDIA GeForce RTX 4050 Laptop GPU 6.0 GB
monai-physio      2026.9.1
monai             1.6.1
totalsegmentator  2.18.0
vtk               9.7.1
trimesh           5.1.1
itk               5.4.7
antspyx           0.6.1


## 2. The monai-physio API, read off the installed copy

The package is young and the docs lag the code, so the names used below are taken
from the installed version rather than from memory.

In [2]:
import monai_physio as mp

public = sorted(n for n in dir(mp) if not n.startswith("_"))
groups: dict[str, list[str]] = {}
for name in public:
    key = (
        "Workflow" if name.startswith("Workflow")
        else "Segment" if name.startswith("Segment")
        else "Register" if name.startswith("Register")
        else "Process" if name.startswith("Process")
        else "other"
    )
    groups.setdefault(key, []).append(name)

for key in ["Workflow", "Segment", "Register", "Process", "other"]:
    names = groups.get(key, [])
    print(f"\n{key} ({len(names)})")
    for name in names:
        print("   ", name)


Workflow (12)
    WorkflowConvertImageToUSD
    WorkflowConvertImageToVTK
    WorkflowConvertVTKToUSD
    WorkflowCreateMeanSurface
    WorkflowCreateStatisticalModel
    WorkflowEvaluateMovement
    WorkflowFinetuneICONRegistration
    WorkflowFitStatisticalModelToPatient
    WorkflowInferMovement
    WorkflowInferPhysicsNeMo
    WorkflowReconstructHighres4DCT
    WorkflowTrainPhysicsNeMo

Segment (7)
    SegmentAnatomyBase
    SegmentChestTotalSegmentator
    SegmentChestTotalSegmentatorWithContrast
    SegmentHeartSimpleware
    SegmentHeartSimplewareTrimmedBranches
    SegmentNVSegmentCT
    SegmentNVSegmentCTMRI

Register (11)
    RegisterImagesANTS
    RegisterImagesBase
    RegisterImagesChain
    RegisterImagesGreedy
    RegisterImagesGreedyICON
    RegisterImagesICON
    RegisterModelsDistanceMaps
    RegisterModelsICP
    RegisterModelsICPITK
    RegisterModelsPCA
    RegisterTimeSeriesImages

Process (9)
    ProcessContours
    ProcessImages
    ProcessLabelmaps
    Process

## 3. Fetch the gated cardiac CT

`Slicer-Heart-CT`: a gated 4D cardiac CT of a truncal valve case, released by the Jolley Lab
at Children's Hospital of Philadelphia through SlicerHeart. Each cardiac phase arrives as its
own `slice_###.mha` volume.

In [3]:
from monai_physio import DownloadData

helpers = [n for n in dir(DownloadData) if n.startswith(("Download", "Verify"))]
print("dataset helpers:")
for name in helpers:
    print("   ", name)
print()
print("Slicer-Heart-CT source:", DownloadData.SLICER_HEART_CT_URL)


dataset helpers:
    DownloadCHOPValve4DData
    DownloadChestCTData
    DownloadKCLHeartModelData
    DownloadPhysicsNeMoMGNLungMotionData
    DownloadSlicerHeartCTData
    DownloadTCIA4DLungData
    VerifyCHOPValve4DData
    VerifyChestCTData
    VerifyDirLab4DCTData
    VerifyKCLHeartModelData
    VerifyPhysicsNeMoMGNLungMotionData
    VerifySlicerHeartCTData
    VerifyTCIA4DLungData

Slicer-Heart-CT source: https://github.com/Project-MONAI/monai-physio/releases/download/2026.07.1/TruncalValve_4DCT.seq.nrrd


In [4]:
SLICER = DATA / "Slicer-Heart-CT"
frames = sorted(SLICER.glob("slice_???.mha"))
if not frames:
    # Downloads TruncalValve_4DCT.seq.nrrd and splits it into one volume per cardiac phase.
    seq_file = DownloadData.DownloadSlicerHeartCTData(SLICER)
    print("sequence:", seq_file.name, f"{seq_file.stat().st_size / 1024**2:.0f} MB")
    frames = sorted(SLICER.glob("slice_???.mha"))

total_mb = sum(f.stat().st_size for f in frames) / 1024**2
print(f"{len(frames)} cardiac phases, {total_mb:.0f} MB total")
for f in frames[:3]:
    print("  ", f.name, f"{f.stat().st_size / 1024**2:.1f} MB")
if len(frames) > 3:
    print("   ...", frames[-1].name)


21 cardiac phases, 1245 MB total
   slice_000.mha 59.9 MB
   slice_001.mha 59.9 MB
   slice_002.mha 59.9 MB
   ... slice_020.mha


## 4. Look at the data before touching it

Geometry of one phase, and a mid-axial slice of each phase written out for the dashboard.

In [5]:
import itk
import numpy as np

volumes = [itk.imread(str(f)) for f in frames]
first = volumes[0]
arr = itk.array_from_image(first)
spacing = np.array(itk.spacing(first))

print("shape (z, y, x) ", arr.shape)
print("spacing mm      ", np.round(spacing, 3))
print("field of view mm", np.round(np.array(arr.shape)[::-1] * spacing, 1))
print("intensity HU    ", int(arr.min()), "to", int(arr.max()))
print("phases          ", len(volumes))

shape (z, y, x)  (218, 391, 512)
spacing mm       [0.6 0.6 0.7]
field of view mm [307.  234.4 152.6]
intensity HU     -1024 to 3071
phases           21


In [6]:
import json
import time

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

EVENTS = DASH / "events.jsonl"


def publish(path: Path, label: str) -> None:
    """Show an image in the local pipeline dashboard."""
    event = {"t": int(time.time() * 1000), "type": "artifact", "path": path.name, "label": label}
    with EVENTS.open("a") as handle:
        handle.write(json.dumps(event) + "\n")


def save_slice(array, name: str, label: str, vmin=-200, vmax=600, overlay=None) -> Path:
    mid = array.shape[0] // 2
    fig, ax = plt.subplots(figsize=(3.2, 3.2), dpi=110)
    ax.imshow(array[mid], cmap="gray", vmin=vmin, vmax=vmax)
    if overlay is not None:
        masked = np.ma.masked_where(overlay[mid] == 0, overlay[mid])
        ax.imshow(masked, cmap="tab20", alpha=0.55, interpolation="nearest")
    ax.axis("off")
    fig.tight_layout(pad=0)
    path = DASH / name
    fig.savefig(path, bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    publish(path, label)
    return path


for index in range(0, len(volumes), max(1, len(volumes) // 6)):
    save_slice(itk.array_from_image(volumes[index]), f"phase_{index:03d}.png", f"phase {index}")
print("previews written to", DASH)

previews written to /mnt/c/Users/vanio/claudeOS/projects/vanioAntunes/.tmp/heart


## 5. Find the heart

`SegmentChestTotalSegmentatorWithContrast` runs TotalSegmentator's nnU-Net over one phase and
returns a labelmap plus one labelmap per anatomy group. Only the reference phase is segmented;
the other 20 phases inherit these labels through registration in the next step.

The reference phase is index 14, which is `int(0.7 * 21)`, the same choice the monai-physio
heart tutorial makes: late diastole, when the chambers are full and their walls are easiest to
separate.


In [7]:
import logging
import time

from monai_physio import SegmentChestTotalSegmentatorWithContrast

REFERENCE = int(0.7 * len(frames))
reference_image = itk.imread(str(frames[REFERENCE]))
print("reference phase:", frames[REFERENCE].name)

segmenter = SegmentChestTotalSegmentatorWithContrast(log_level=logging.INFO)
segmenter.set_fast_mode(False)  # full 1.5 mm model: 400 s against 42 s, but a sharper label

started = time.time()
result = segmenter.segment(reference_image)
print()
print(f"segmented in {time.time() - started:.0f} s")
print("groups returned:", sorted(result.keys()))


reference phase: slice_014.mha


2026-10-07 20:05:14 WARNING SegmentChestTotalSegmentatorWithContrast The input image should have isotropic spacing


2026-10-07 20:05:14 INFO SegmentChestTotalSegmentatorWithContrast Input image has spacing: itkVectorD3 ([0.599609, 0.599609, 0.7])


2026-10-07 20:05:14 INFO SegmentChestTotalSegmentatorWithContrast Resampling to isotropic: 1.000



If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024



...

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 19.37it/s]

Resampling...


  Resampled in 0.61s



segmented in 110 s
groups returned: ['bone', 'contrast', 'heart', 'labelmap', 'lung', 'major_vessels', 'other', 'soft_tissue']


In [8]:
# What the network actually found, as volumes in millilitres.
CHAMBERS = {
    141: "left atrium",
    142: "left ventricle",
    143: "right atrium",
    144: "right ventricle",
}

labelmap = result["labelmap"]
labels = itk.array_from_image(labelmap)
voxel_ml = float(np.prod(itk.spacing(labelmap))) / 1000.0

present, counts = np.unique(labels[labels > 0], return_counts=True)
print(f"{len(present)} structures found, voxel = {voxel_ml * 1000:.3f} mm3")
print()
for label_id, count in sorted(zip(present.tolist(), counts.tolist()), key=lambda pair: -pair[1])[:12]:
    print(f"  label {label_id:>4}  {count * voxel_ml:8.1f} mL  {CHAMBERS.get(label_id, '')}")

print()
print("chambers:")
for label_id, name in CHAMBERS.items():
    volume = float((labels == label_id).sum()) * voxel_ml
    print(f"  {name:<16} {volume:7.1f} mL")


58 structures found, voxel = 0.252 mm3

  label   14     672.4 mL  
  label  155     562.5 mL  
  label   11     511.8 mL  
  label    5     486.3 mL  
  label   10     433.1 mL  
  label   13     300.8 mL  
  label   12     248.7 mL  
  label   51     215.7 mL  
  label   87      84.2 mL  
  label   86      77.9 mL  
  label    6      65.1 mL  
  label  122      56.7 mL  

chambers:
  left atrium          0.0 mL
  left ventricle       0.0 mL
  right atrium         0.0 mL
  right ventricle      0.0 mL


In [9]:
# Save the labelmap, and show it on top of the CT so the result can be judged by eye.
OUT_T1 = OUT / "tutorial_01_heart"
OUT_T1.mkdir(parents=True, exist_ok=True)
labelmap_path = OUT_T1 / f"slice_{REFERENCE:03d}_labelmap.mha"
itk.imwrite(labelmap, str(labelmap_path), compression=True)
print("written:", labelmap_path, f"{labelmap_path.stat().st_size / 1024**2:.1f} MB")

ct = itk.array_from_image(reference_image)
save_slice(ct, f"seg_{REFERENCE:03d}_all.png", f"phase {REFERENCE}: all structures", overlay=labels)

chambers_only = np.where(np.isin(labels, list(CHAMBERS)), labels, 0)
save_slice(ct, f"seg_{REFERENCE:03d}_chambers.png", f"phase {REFERENCE}: chambers only", overlay=chambers_only)


written: /root/heart/out/tutorial_01_heart/slice_014_labelmap.mha 1.5 MB


PosixPath('/mnt/c/Users/vanio/claudeOS/projects/vanioAntunes/.tmp/heart/seg_014_chambers.png')

## 6. What the free weights can and cannot see

TotalSegmentator returns the heart as one structure. The six cardiac substructures
(`highres_myocardium`, `highres_atrium_left`, `highres_ventricle_left`,
`highres_atrium_right`, `highres_ventricle_right`, `highres_pulmonary_artery`) come from the
`heartchambers_highres` task, which the authors gate behind a licence key that is free for
academic use on request. NVIDIA's NV-Segment-CT has the same limitation, so this is a property
of the open weights rather than of this backend.

The run below therefore builds the whole heart and the great vessels. When a key is installed,
`segmenter.set_has_academic_license(True)` is the only change needed and the rest of the
pipeline is unchanged.


In [10]:
for backend in (segmenter,):
    heart_labels = backend.taxonomy.labels_in_group("heart")
    print(type(backend).__name__)
    for label_id, name in sorted(heart_labels.items()):
        volume = float((labels == label_id).sum()) * voxel_ml
        status = f"{volume:7.1f} mL" if volume else "    none, needs licence"
        print(f"  {label_id:>4}  {name:<26} {status}")


SegmentChestTotalSegmentatorWithContrast
    51  heart                        215.7 mL
    61  atrial_appendage_left          1.1 mL
   140  highres_myocardium             none, needs licence
   141  highres_atrium_left            none, needs licence
   142  highres_ventricle_left         none, needs licence
   143  highres_atrium_right           none, needs licence
   144  highres_ventricle_right        none, needs licence
   146  highres_pulmonary_artery       none, needs licence


### First attempt: killed by the out-of-memory killer

Running the workflow on all 21 phases at the native 0.6 by 0.6 by 0.7 mm reached 23.9 GB of
resident memory at phase 6 and the Linux OOM killer took the process:

```
Out of memory: Killed process 4683 (python) anon-rss:23884420kB
```

The fix costs no accuracy. `SegmentChestTotalSegmentatorWithContrast` already resamples its
input to 1.0 mm isotropic before inference, so the extra sampling density was being discarded
anyway. Resampling the whole series to 1.0 mm first cuts the voxel count, and the memory with
it, by a factor of about four.


In [11]:
def to_isotropic(image, spacing_mm=1.0):
    """Resample an ITK image to isotropic spacing, keeping physical extent."""
    in_spacing = np.array(itk.spacing(image))
    in_size = np.array(itk.size(image))
    out_size = np.round(in_size * in_spacing / spacing_mm).astype(int)
    return itk.resample_image_filter(
        image,
        interpolator=itk.LinearInterpolateImageFunction.New(image),
        size=[int(v) for v in out_size],
        output_spacing=[float(spacing_mm)] * 3,
        output_origin=itk.origin(image),
        output_direction=image.GetDirection(),
        default_pixel_value=-1024,
    )


started = time.time()
volumes_1mm = [to_isotropic(volume, 1.0) for volume in volumes]
voxels_before = np.prod(itk.size(volumes[0]))
voxels_after = np.prod(itk.size(volumes_1mm[0]))
print("resampled", len(volumes_1mm), "phases in", f"{time.time() - started:.0f} s")
print("size per phase:", list(itk.size(volumes[0])), "->", list(itk.size(volumes_1mm[0])))
print("voxels per phase:", f"{voxels_before/1e6:.1f} M -> {voxels_after/1e6:.1f} M",
      f"({voxels_after / voxels_before:.2f} of the original)")
print("series in memory:", f"{voxels_after * 2 * len(volumes_1mm) / 1024**3:.1f} GB as int16")


resampled 21 phases in 1 s
size per phase: [512, 391, 218] -> [307, 234, 153]
voxels per phase: 43.6 M -> 11.0 M (0.25 of the original)
series in memory: 0.4 GB as int16


## 7. Measure the motion and build the surfaces

`WorkflowConvertImageToUSD` is the single call that does the rest: it segments the reference
phase, registers every other phase onto it with greedy diffeomorphic registration, carries the
labels across, extracts a surface per structure per phase, and writes an animated USD.

Greedy runs on the CPU over 4 resolution levels. With 21 phases this is the long stage of the
pipeline, which is why the earlier memory decision mattered.


In [12]:
from monai_physio import RegisterImagesGreedy, WorkflowConvertImageToUSD

registrar = RegisterImagesGreedy(log_level=logging.INFO)
registrar.set_number_of_iterations([30, 15, 7, 3])
registrar.set_metric("CC")

workflow = WorkflowConvertImageToUSD(
    time_series_images=volumes_1mm,
    reference_image=volumes_1mm[REFERENCE],
    output_directory=str(OUT_T1),
    usd_project_name="cardiac_model",
    registration_method=registrar,
    segmentation_method=segmenter,
    surface_reduction_rate=0.5,
    log_level=logging.INFO,
    save_assets=True,
)

started = time.time()
workflow_results = workflow.process()
print()
print(f"workflow finished in {(time.time() - started) / 60:.1f} min")
print("results:", workflow_results)
print("dynamic labelmap ids:", workflow.dynamic_labelmap_ids)


2026-10-07 20:07:06 INFO WorkflowConvertImageToUSD ======================================================================


2026-10-07 20:07:06 INFO WorkflowConvertImageToUSD Image-to-USD Processing Pipeline


2026-10-07 20:07:06 INFO WorkflowConvertImageToUSD ======================================================================


2026-10-07 20:07:06 INFO WorkflowConvertImageToUSD Segmenting and registering frames...


2026-10-07 20:07:06 INFO WorkflowConvertImageToUSD Segmenting reference image...


...

2026-10-07 20:25:50 INFO Converting 21 meshes to /root/heart/out/tutorial_01_heart/cardiac_model.all.usd


2026-10-07 20:26:53 INFO Saved USD file: /root/heart/out/tutorial_01_heart/cardiac_model.all.usd


2026-10-07 20:26:53 INFO WorkflowConvertImageToUSD Painting all anatomy USD...


2026-10-07 20:26:54 INFO WorkflowConvertImageToUSD Processing pipeline completed successfully



workflow finished in 19.8 min
results: {'all': 'cardiac_model.all_painted.usd'}
dynamic labelmap ids: []


In [13]:
# What landed on disk, and how big the web payload would be if shipped raw.
produced = sorted(OUT_T1.rglob("*"))
by_kind = {}
for path in produced:
    if path.is_file():
        by_kind.setdefault(path.suffix, []).append(path)

for suffix, paths in sorted(by_kind.items(), key=lambda kv: -sum(p.stat().st_size for p in kv[1])):
    size_mb = sum(p.stat().st_size for p in paths) / 1024**2
    print(f"{suffix or '(none)':<8} {len(paths):>4} files  {size_mb:8.1f} MB")

usd_name = workflow_results.get("dynamic") or workflow_results.get("all")
if usd_name:
    usd_file = OUT_T1 / usd_name
    print()
    print("animated model:", usd_file.name, f"{usd_file.stat().st_size / 1024**2:.1f} MB")


.hdf       42 files    4331.9 MB
.usd        2 files     471.5 MB
.mha       23 files     314.4 MB

animated model: cardiac_model.all_painted.usd 235.7 MB


## 8. From USD to a web page

The workflow's output is a 236 MB animated USD, built for Omniverse. Three scripts finish the
job, each kept out of this notebook so they can be rerun on their own:

- `scripts/heart/probe_usd.py` reports what the USD holds: 55 meshes, each with 21 time samples
  of per-vertex positions, stage units in metres.
- `scripts/heart/measure_volumes.py` checks whether a volume can honestly be computed from
  those surfaces. It cannot: the heart surface has **1770 open boundary edges** where the field
  of view truncates it, so the volume it encloses is undefined. The script refuses to write
  numbers when that check fails, which is why the page publishes no volumes and no ejection
  fraction. An earlier attempt did publish them, and they moved by tens of per cent between two
  segmentations of the same heart.
- `scripts/heart/export_gltf.py` writes `imaging/heart/heart.glb`: decimation that preserves
  vertex correspondence across phases, Taubin smoothing, base-pose normals, and glTF morph
  targets so the browser interpolates between phases.

### What was dropped, and why

The `contrast` structure was shipped at first as a "blood pool". It is a contrast-agent
detection map rather than an anatomical cast: it reaches the edge of the scan, breaks into
sheets, and its enclosed volume is not stable between runs. It is no longer exported.

The fast 3 mm segmentation was replaced by the full 1.5 mm model after
`scripts/heart/compare_segmenters.py` measured both on the same phase: 42 s against 400 s,
55 structures against 58, and a visibly coarser wall. Only the reference phase is segmented, so
the cost is paid once.
